# Paid orders by status

**This lab is done by a team of three, on one work laptop.** The README says how the three roles work. Write your
names in the next cell.

The monthly board pack needs two things: the total that customers paid, and, for each order status, the number of
paid orders and the average order value. A colleague has written a report for it. **You will meet that report in
section D.** First your team writes queries that work: a look at the payments, a join of its own with its check, and
the orders with more than one payment row.

The data is a selected sample of the Olist marketplace (Brazil; money is in BRL), one CSV per table in `data/raw/`.
This lab uses two of them: `orders` (the orders, with their status) and `order_payments` (the payments made for
them).

Two definitions, from finance. They are what the board pack's words mean:

> A **paid order** is an order with at least one payment row in `order_payments`.
>
> **Average order value** is what was paid divided by the number of paid orders, both over the same orders.

The definition counts every status: a canceled order with a payment row is a paid order too. An order can be paid in
several parts (a card and some vouchers, say), and each part is its own row of `order_payments`.

**Then run the three code cells below.** The first prints the folder it is working in: it must be this project's
folder. The second makes each file a view with the file's name. The third gets the checks ready. Then work top to bottom, **running each cell as you
come to it**.

## Your team

**Keyboard:** ______ · **AI:** ______ · **Checker:** ______

Everyone's name, beside their role (a team of two: the AI person is the checker too; a team of four: two names beside
Checker). This line is the lab's checkpoint. Before you leave, the keyboard uploads this notebook to Moodle; everyone
else pastes this line as their submission text.

In [ ]:
import os
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 400)   # up to 400 rows in full; a longer result prints head and tail with "..." between

# Anchor to the project folder (DS1, Block 1), then stand there: every path below is from the project folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
print("Working in:", Path.cwd())   # must print this project's folder

con = duckdb.connect()             # an in-memory database; it reads the files in data/raw/ directly

In [ ]:
# Every table as a view with its file's name, so a query can say FROM orders instead of the path.
# A view is a saved query (Lab 1): nothing is copied; the file in data/raw/ is read each time.
TABLES = ["orders", "order_payments"]
for t in TABLES:
    con.sql(f"CREATE OR REPLACE VIEW {t} AS SELECT * FROM 'data/raw/{t}.csv'")
print("views:", ", ".join(TABLES))

In [ ]:
# The checks, written for you. Run this cell once, after the views; you do not need to read it.
# Each section's check cell calls one of these functions. Every one computes its numbers from the tables again.
def check(ok, claim, evidence=""):
    """Every check prints its lines with this: PASS or FAIL, what was tested, then the numbers it used."""
    print("PASS" if ok else "FAIL", f"  {claim:<42} {evidence}")


def ready(needs):
    """True when each view a check reads exists, runs, and has its columns. If not, one FAIL line says what to do."""
    for view, columns in needs.items():
        if not con.sql(f"SELECT 1 FROM duckdb_views() WHERE lower(view_name) = '{view}'").fetchall():
            check(False, f"no view named {view} yet", "write your query above, and run it")
            return False
        try:
            have = [r[0] for r in con.sql(f"DESCRIBE {view}").fetchall()]
            # One row is enough to see that it runs: a wrong join can have millions.
            con.sql(f"SELECT * FROM {view} LIMIT 1").fetchall()
        except duckdb.Error as e:
            check(False, f"your view {view} does not run")
            print(f"       {str(e).splitlines()[0]}")
            print("       fix the cell that makes it, and run that cell again")
            return False
        missing = [c for c in columns if c not in [h.lower() for h in have]]
        if missing:
            check(False, f"{view} has no column {missing[0]}", "its columns: " + ", ".join(have))
            print(f"       name the column with AS {missing[0]} in your query")
            return False
    return True


def shows_errors(f):
    """A check that cannot read a view says so in a FAIL line, not in a Python error."""
    def run():
        try:
            f()
        except duckdb.Error as e:
            check(False, "a check could not read your view")
            print(f"       {str(e).splitlines()[0]}")
            print("       read the columns of your view: are the numbers numbers?")
    return run


@shows_errors
def check_b():
    """Section B: your join, paid_by_status, against order_payments and orders."""
    if not ready({'paid_by_status': ['order_status', 'payment_rows', 'paid']}):
        return
    # The check after a join (Block 3, cell 2): rows in, rows out, distinct orders out; then the money and the shape.
    rows_in   = con.sql("SELECT COUNT(*) FROM order_payments").fetchone()[0]
    rows_out  = con.sql("SELECT COUNT(*) FROM order_payments AS p JOIN orders AS o ON o.order_id = p.order_id").fetchone()[0]
    orders_in = con.sql("SELECT COUNT(DISTINCT order_id) FROM order_payments").fetchone()[0]
    orders_out = con.sql("""
        SELECT COUNT(DISTINCT p.order_id) FROM order_payments AS p JOIN orders AS o ON o.order_id = p.order_id
    """).fetchone()[0]
    rows_yours    = int(con.sql("SELECT COALESCE(SUM(payment_rows), 0) FROM paid_by_status").fetchone()[0])
    paid_in_table = float(con.sql("SELECT COALESCE(SUM(paid), 0) FROM paid_by_status").fetchone()[0])
    total_paid    = con.sql("SELECT SUM(payment_value) FROM order_payments").fetchone()[0]
    # Status by status: the payment rows of the orders with that status, and what they add up to, picked out with IN.
    off_rows = con.sql("""
        SELECT COUNT(*) FROM paid_by_status AS mine
        LEFT JOIN (SELECT s.order_status, COUNT(*) AS payment_rows, SUM(p.payment_value) AS paid
                   FROM (SELECT DISTINCT order_status FROM orders) AS s, order_payments AS p
                   WHERE p.order_id IN (SELECT order_id FROM orders WHERE order_status = s.order_status)
                   GROUP BY s.order_status) AS src ON src.order_status = CAST(mine.order_status AS VARCHAR)
        WHERE src.paid IS NULL OR mine.payment_rows != src.payment_rows OR abs(mine.paid - src.paid) >= 0.005
    """).fetchone()[0]
    # The shape: one row per status that paid orders have (counted in orders, with no join), each a real status.
    rows_in_table = con.sql("SELECT COUNT(*) FROM paid_by_status").fetchone()[0]
    statuses      = con.sql("SELECT COUNT(DISTINCT order_status) FROM paid_by_status").fetchone()[0]
    statuses_paid = con.sql("""
        SELECT COUNT(DISTINCT order_status) FROM orders WHERE order_id IN (SELECT order_id FROM order_payments)
    """).fetchone()[0]
    not_a_status  = con.sql("""
        SELECT COUNT(*) FROM paid_by_status
        WHERE NOT COALESCE(CAST(order_status AS VARCHAR) IN (SELECT order_status FROM orders), FALSE)
    """).fetchone()[0]
    # The three counts (Block 3's cell 2): the rows in, the rows your join kept, the distinct orders out.
    check(rows_in == rows_out == rows_yours and orders_in == orders_out, "every payment joined to its order, once",
          f"{rows_in:,} in; payment_rows add to {rows_yours:,}; {orders_out:,} orders")
    check(abs(paid_in_table - total_paid) < 0.005, "the money agrees to the cent", f"{paid_in_table:,.2f} BRL")
    check(off_rows == 0, "each status's rows and money are its own", f"{off_rows:,} rows are off")
    check(rows_in_table == statuses == statuses_paid and not_a_status == 0, "one row per status, each a real one",
          f"{rows_in_table:,} rows, {statuses_paid:,} statuses paid; {not_a_status:,} are not a status")


@shows_errors
def check_c1():
    """Section C1: your several_payments, order by order, against order_payments."""
    if not ready({'several_payments': ['order_id', 'payment_rows']}):
        return
    # Your several_payments, against order_payments: the orders with more than one payment row, and their rows.
    mine_orders = con.sql("SELECT COUNT(*) FROM several_payments").fetchone()[0]
    mine_ids    = con.sql("SELECT COUNT(DISTINCT order_id) FROM several_payments").fetchone()[0]
    mine_rows   = int(con.sql("SELECT COALESCE(SUM(payment_rows), 0) FROM several_payments").fetchone()[0])
    ones        = con.sql("SELECT COUNT(*) FROM several_payments WHERE NOT COALESCE(payment_rows > 1, FALSE)").fetchone()[0]
    rows        = con.sql("SELECT COUNT(*) FROM order_payments").fetchone()[0]
    orders_     = con.sql("SELECT COUNT(DISTINCT order_id) FROM order_payments").fetchone()[0]
    # Order by order, from your rows: is it an order with more than one payment row, and is its count right?
    off = con.sql("""
        SELECT COUNT(*) FROM several_payments AS mine
        LEFT JOIN (SELECT order_id, COUNT(*) AS payment_rows FROM order_payments GROUP BY order_id) AS src
               ON src.order_id = mine.order_id
        WHERE src.order_id IS NULL OR src.payment_rows <= 1 OR mine.payment_rows IS DISTINCT FROM src.payment_rows
    """).fetchone()[0]
    need = con.sql("""
        SELECT COUNT(*) FROM (SELECT order_id FROM order_payments GROUP BY order_id HAVING COUNT(*) > 1)
    """).fetchone()[0]
    check(mine_orders == mine_ids == need and off == 0, "every order paid in several parts, once",
          f"{mine_orders} rows, {mine_ids} orders, {need} needed; {off} off")
    check(ones == 0, "no order with one payment row", f"{ones} have one")
    extra = mine_rows - mine_orders
    check(orders_ + extra == rows, "they are the whole gap",
          f"{mine_rows} - {mine_orders} = {extra}; {orders_:,} + {extra} = {orders_ + extra:,} rows")


@shows_errors
def check_c2():
    """Section C2: your top_payments, row by row, against order_payments."""
    if not ready({'top_payments': ['order_id', 'payment_sequential', 'payment_type', 'payment_value']}):
        return
    # Your C2 join, row by row from your rows: a real payment of an order with 9 or more rows, unchanged, and all of them.
    top_need = con.sql("""
        SELECT COUNT(*) FROM order_payments
        WHERE order_id IN (SELECT order_id FROM order_payments GROUP BY order_id HAVING COUNT(*) >= 9)
    """).fetchone()[0]
    top_rows = con.sql("SELECT COUNT(*) FROM top_payments").fetchone()[0]
    top_keys = con.sql("SELECT COUNT(DISTINCT (order_id, payment_sequential)) FROM top_payments").fetchone()[0]
    top_off  = con.sql("""
        SELECT COUNT(*) FROM top_payments AS mine
        LEFT JOIN order_payments AS p ON p.order_id = mine.order_id AND p.payment_sequential = mine.payment_sequential
        WHERE p.order_id IS NULL OR mine.payment_value IS DISTINCT FROM p.payment_value
           OR mine.payment_type IS DISTINCT FROM p.payment_type
           OR p.order_id NOT IN (SELECT order_id FROM order_payments GROUP BY order_id HAVING COUNT(*) >= 9)
    """).fetchone()[0]
    check(top_rows == top_keys == top_need and top_off == 0, "C2: every payment of the 9-or-more orders",
          f"{top_rows} rows, {top_need} needed; {top_off} off")


@shows_errors
def check_e():
    """Section E: the evidence, then five lines about your status_summary."""
    if not ready({'status_summary': ['order_status', 'orders', 'avg_order_value', 'paid']}):
        return
    rows_in_table   = con.sql("SELECT COUNT(*) FROM status_summary").fetchone()[0]
    statuses        = con.sql("SELECT COUNT(DISTINCT order_status) FROM status_summary").fetchone()[0]
    no_status       = con.sql("SELECT COUNT(*) FROM status_summary WHERE order_status IS NULL").fetchone()[0]
    # Is every value in the order_status column a status that orders has? A NULL, or anything else, is counted.
    not_a_status    = con.sql("""
        SELECT COUNT(*) FROM status_summary
        WHERE NOT COALESCE(CAST(order_status AS VARCHAR) IN (SELECT order_status FROM orders), FALSE)
    """).fetchone()[0]
    # How many statuses must the table have? The statuses of the orders that have a payment row, counted without a join.
    statuses_paid   = con.sql("""
        SELECT COUNT(DISTINCT order_status) FROM orders
        WHERE order_id IN (SELECT order_id FROM order_payments)
    """).fetchone()[0]
    orders_in_table = int(con.sql("SELECT COALESCE(SUM(orders), 0) FROM status_summary").fetchone()[0])
    paid_in_table   = float(con.sql("SELECT COALESCE(SUM(paid), 0) FROM status_summary").fetchone()[0])
    paid_orders     = con.sql("SELECT COUNT(DISTINCT order_id) FROM order_payments").fetchone()[0]
    total_paid      = con.sql("SELECT SUM(payment_value) FROM order_payments").fetchone()[0]
    # A status passes only if its average is there and agrees with paid / orders to the cent. A missing average, or a
    # status with no orders, makes the comparison NULL, and COALESCE counts that as not agreeing.
    off_average     = con.sql("""
        SELECT COUNT(*) FROM status_summary
        WHERE NOT COALESCE(abs(avg_order_value - paid / orders) < 0.005, FALSE)
    """).fetchone()[0]

    # Status by status: the paid orders of that status, counted in orders with no join, and what they paid, added up from
    # order_payments. A row of yours whose orders or money is not the source's is counted.
    off_orders, off_money = con.sql("""
        SELECT COUNT(*) FILTER (WHERE src.orders IS NULL OR mine.orders != src.orders),
               COUNT(*) FILTER (WHERE src.paid IS NULL OR abs(mine.paid - src.paid) >= 0.005)
        FROM status_summary AS mine
        LEFT JOIN (SELECT o.order_status,
                          COUNT(*) FILTER (WHERE o.order_id IN (SELECT order_id FROM order_payments)) AS orders,
                          (SELECT SUM(p.payment_value) FROM order_payments AS p
                           JOIN orders AS o2 ON o2.order_id = p.order_id WHERE o2.order_status = o.order_status) AS paid
                   FROM orders AS o GROUP BY o.order_status) AS src ON src.order_status = CAST(mine.order_status AS VARCHAR)
    """).fetchone()
    # Orders in the table that have no payment row at all (a repair that starts FROM orders and keeps every order).
    unpaid_rows     = con.sql("""
        SELECT COUNT(*) FROM orders AS o LEFT JOIN order_payments AS p ON p.order_id = o.order_id WHERE p.order_id IS NULL
    """).fetchone()[0]

    print("Evidence, with no join:")
    print(f"   order_payments: {paid_orders:,} distinct orders, {total_paid:,.2f} BRL paid")
    print(f"   orders:         {con.sql('SELECT COUNT(*) FROM orders').fetchone()[0]:,} rows")
    print(f"   your status_summary: {rows_in_table:,} rows, {orders_in_table:,} orders, {paid_in_table:,.2f} BRL"
          + (f"; {paid_in_table / orders_in_table:,.2f} BRL per order over all" if orders_in_table else ""))
    check(rows_in_table == statuses == statuses_paid and no_status == 0, "[1] one row per status",
          f"{rows_in_table:,} rows, {statuses_paid:,} statuses have paid orders")
    check(not_a_status == 0, "[2] each order_status is a real status", f"{not_a_status:,} are not")
    check(orders_in_table == paid_orders and off_orders == 0, "[3] its orders are the paid orders",
          f"{orders_in_table:,} here, {paid_orders:,} paid; {off_orders:,} rows are off")
    if orders_in_table > paid_orders and orders_in_table - paid_orders == unpaid_rows:
        print(f"       ({unpaid_rows} order(s) in your table have no payment row at all: a paid order has at least one)")
    check(abs(paid_in_table - total_paid) < 0.005 and off_money == 0, "[4] its money is the total paid",
          f"{paid_in_table:,.2f} BRL; {off_money:,} rows are off")
    check(off_average == 0, "[5] each average is paid / orders", f"{off_average:,} are off")


print("checks ready: check_b, check_c1, check_c2, check_e")

---

# Your work starts here

Work top to bottom, as a team: one keyboard, one AI, one checker (README, *How your team works*). **For the first
five minutes the AI person shares nothing from the assistant**, and their own ideas from the first minute.

Each section names the lecture cells it uses (`notebooks/lecture.ipynb`, numbered as the slides called them; the
Block 3 cells it cites are copied at the end of that notebook). Run each cell as you come to it. Every check prints
`PASS` or `FAIL` on the left, its numbers on the right, and computes them from the tables. **A check reads your
view as it was last made: after you edit a query, run its cell again, without an error, then run the check.** The
views you make live in memory, so after a kernel restart run the cells again from the top down to the one you had
reached, and no further.

## A. Look first

*Session 1: `LIMIT`, the counts and `GROUP BY` (Block 2). No join yet.*

**Say it before you type it:** for each query, the two of you who are not at the keyboard say what it is; then the
keyboard types what the team agrees on.

1. Five rows of `order_payments`, `SELECT *` with `LIMIT 5`. Say out loud what one row is, and which column it shares
   with `orders`.
2. Supplied: how many orders each `order_status` has, most orders first. Run it and read it.
3. How many rows does `order_payments` have, and what do its `payment_value`s add up to?

Checker, before each one runs: how many rows will come back, and why?

**What you should see:** in 2, six statuses, `delivered` first, with 2,555 orders, adding up to 2,653 orders in all;
in 3, one row, 2,790 payment rows adding up to 435,300.95 BRL. If yours differ, say the query again before going
on.

In [ ]:
# A1. Five rows of order_payments.

In [ ]:
# A2, supplied. One row per status: six.
display(con.sql("""
    SELECT order_status, COUNT(*) AS orders
    FROM orders
    GROUP BY order_status
    ORDER BY orders DESC
""").df())
print(f"in all: {con.sql('SELECT COUNT(*) FROM orders').fetchone()[0]:,} orders")

In [ ]:
# A3. The number of rows in order_payments, and what payment_value adds up to.

## B. Your join: what was paid, by order status

*Block 3, cells 1 to 3: a join, its three counts, and a join followed by `GROUP BY`.*

The status is on `orders`; the money is on `order_payments`. Write it in the empty cell below: every payment joined
to its order (every one: no `WHERE`), then what was paid, by order status, saved as a view named `paid_by_status`
with the columns `order_status`, `payment_rows` (how many payment rows each status has: `COUNT(*)`) and `paid`;
then show it, most paid first. Start `FROM order_payments AS p`: the money is on its rows.

**Say the `JOIN` line out loud before the keyboard types it.** **Checker, before the check runs:** `order_payments`
has 2,790 rows (A3). How many rows must come out of the join? What must the money add up to?
**After the check runs, the checker says the answer out loud:** which status was paid the most, and how much? And
which was paid the least? Then say why this join gets that question right, and write one comment: before the `GROUP BY`,
what is one row of the join?

In [ ]:
# B. Your join, saved as the view paid_by_status; then the view, most paid first.

Its check is written for you, in the cell *The checks* near the top: run the cell below, and say which line failed. Four lines must say `PASS`.

In [ ]:
# The check itself is written for you, in the cell *The checks* near the top.
check_b()

In [ ]:
# Before the GROUP BY, one row of this join is one ...

## C. The orders with more than one payment row

*Lecture cells 4 and 6 (`HAVING`: the keys that repeat); Block 3, cell 1 (a join).*

Section B's check counted 2,790 payment rows and 2,652 distinct orders. So some orders have more than one row in
`order_payments`. Which ones?

1. **In C1, the first empty cell:** a view named `several_payments`, one row for **every** order that has more than one
   payment row, with its `order_id` and its number of payment rows as `payment_rows` (count rows with `COUNT(*)`: not
   `payment_installments`, not the largest `payment_sequential`). Then, in the same cell, a second query
   shows five rows, `ORDER BY payment_rows DESC, order_id`. **The `LIMIT 5` goes in that second query, never in the view.** Lecture cell 4 is this query on another table. **Say the `GROUP BY` and
   the `HAVING` out loud before the keyboard types them.** DuckDB would also accept `payment_rows` in the `HAVING`;
   most databases do not, so write the `COUNT(*)` again, as the lecture did.
2. **In C2, the second empty cell, a join:** a view named `top_payments`, every payment row of the orders in your view
   that have at least 9 payment rows, from `several_payments` joined to `order_payments`: `order_id`, `payment_sequential`,
   `payment_type` and `payment_value`. Then show it, `ORDER BY order_id, payment_sequential`. Pick one order: what do its rows say about how it was
   paid?

**Checker, before C1's check runs:** can any `payment_rows` in your view be 1? And for all the orders in
`several_payments`, how many payment rows are there beyond one per order?

In [ ]:
# C1. The view several_payments; then the five orders with the most payment rows.

C1's check is written for you: run the cell below once `several_payments` exists, before you write C2. Three lines must say `PASS`; the third is the arithmetic that connects your view to section B's two counts. **Checker:** read it out.

In [ ]:
# The check itself is written for you, in the cell *The checks* near the top.
check_c1()

In [ ]:
# C2. The view top_payments: every payment row of the orders in several_payments with 9 or more. Then show it.

C2's check: run the cell below once `top_payments` exists. Its line must say `PASS`.

In [ ]:
# The check itself is written for you, in the cell *The checks* near the top.
check_c2()

## D. Your colleague's report: the same join, a different question

*Lecture cells 2 and 3 (what one row is after a join); cells 7 and 8 (`WITH`: one row per order first, then join).*

Now the report. Your colleague wrote the next two cells for the board pack. They run without an error. **Run them
and read the numbers.** The first is the total that customers paid. The second is the table the board asked for:
for each order status, the number of paid orders and the average order value.

**Checker, before the second one runs:** it is the join you checked in section B. What is one row of that join? Then
what will `COUNT(*)` count?

### The report, first cell: total paid

In [ ]:
total_paid = con.sql("SELECT SUM(payment_value) FROM order_payments").fetchone()[0]
print(f"Total paid: {total_paid:,.2f} BRL")

### The report, second cell: paid orders and average order value, by order status

In [ ]:
con.sql("""
    CREATE OR REPLACE VIEW report_by_status AS
    SELECT o.order_status,
           COUNT(*)             AS orders,
           AVG(p.payment_value) AS avg_order_value,
           SUM(p.payment_value) AS paid
    FROM order_payments AS p
    JOIN orders AS o ON o.order_id = p.order_id
    GROUP BY o.order_status
""")
display(con.sql("""
    SELECT order_status, orders, ROUND(avg_order_value, 2) AS avg_order_value, ROUND(paid, 2) AS paid
    FROM report_by_status
    ORDER BY orders DESC
""").df())
print(f"Paid orders in this table: {con.sql('SELECT SUM(orders) FROM report_by_status').fetchone()[0]:,}")
print(f"Paid in this table:        {con.sql('SELECT SUM(paid) FROM report_by_status').fetchone()[0]:,.2f} BRL")

The report says how many paid orders its table holds. The cell below is supplied: it puts that number beside the rows
of `orders` and two counts of `order_payments`. By the definition, the paid orders are the different `order_id`s in
`order_payments`.

In [ ]:
report_orders = int(con.sql("SELECT COALESCE(SUM(orders), 0) FROM report_by_status").fetchone()[0])
order_rows    = con.sql("SELECT COUNT(*) FROM orders").fetchone()[0]
payment_rows  = con.sql("SELECT COUNT(*) FROM order_payments").fetchone()[0]
paid_orders   = con.sql("SELECT COUNT(DISTINCT order_id) FROM order_payments").fetchone()[0]
print(f"the report's orders column adds up to: {report_orders:,}")
print(f"rows in orders:                        {order_rows:,}")
print(f"rows in order_payments:                {payment_rows:,}")
print(f"distinct order_id in order_payments:   {paid_orders:,}")

**Checker, read the two numbers out, before anything changes:** 2,790 paid orders in the report, and `orders` has
2,653 rows (your A2 table adds up to it). Paid orders cannot be more than orders. The money, 435,300.95, is the
total paid (A3): `SUM` was not hurt by the join.

**Why?** Section C has what explains it. Say it to the team: what did `COUNT(*)` count, and what did
`AVG(p.payment_value)` average? Then say what would happen if you changed only `COUNT(*)` to
`COUNT(DISTINCT p.order_id)`: which column would still be wrong?

### The repair: one row per order first

**Say it first:** what must one row be, *before* the join to `orders`, so that `COUNT(*)` counts orders and `AVG`
averages orders?

The cell below runs as it is. It is your colleague's query under a new name, `status_summary`. **Rewrite it** so
that both the `orders` column and the average are right. Try your way first. The way the lecture showed: one row per
order before the join.

*If you want the steps:*

1. in a `WITH`, make `order_payments` one row per order: the `order_id`, and what was paid for it in all, a `SUM`
   (lecture cell 7 made one row per order with `COUNT(*)`; cell 10 with a `SUM`);
2. join that to `orders`;
3. group by `order_status`, with the same three columns: `orders`, `avg_order_value`, `paid`.

A view of your own, made the same way, in place of the `WITH` is right too: the check reads only `status_summary`.

Once one row is one order, `AVG` over the rows is the total paid divided by the orders: the definition. Do not round
inside the view: the second query rounds for display, and the two lines under it add up the `orders` and `paid`
columns for you. **The comment on the first line is yours: one sentence on why
the report counted 2,790.** The checker says it; the keyboard types it. Checker, before it runs: what must the
`orders` column add up to now? And `paid`?

In [ ]:
con.sql("""
    -- Why the report counted 2,790 paid orders: ________
    CREATE OR REPLACE VIEW status_summary AS
    SELECT o.order_status,
           COUNT(*)             AS orders,
           AVG(p.payment_value) AS avg_order_value,
           SUM(p.payment_value) AS paid
    FROM order_payments AS p                       -- your colleague's query: rewrite it
    JOIN orders AS o ON o.order_id = p.order_id
    GROUP BY o.order_status
""")
display(con.sql("""
    SELECT order_status, orders, ROUND(avg_order_value, 2) AS avg_order_value, ROUND(paid, 2) AS paid
    FROM status_summary
    ORDER BY orders DESC
""").df())
print(f"orders adds up to: {con.sql('SELECT SUM(orders) FROM status_summary').fetchone()[0]:,}")
print(f"paid adds up to:   {con.sql('SELECT SUM(paid) FROM status_summary').fetchone()[0]:,.2f} BRL")

## E. The check

*Lecture cell 9 (the three counts after the repair); Block 2, "to the penny".*

Supplied: its cell is one line, and the check is in *The checks* near the top. It reads your `status_summary`. First the evidence: the paid orders and the total paid in `order_payments`,
computed with no join, then your table's overall average. Then five lines about your table, numbered `[1]` to
`[5]`: one row per status; every value in its `order_status` column a status that `orders` has; its orders are the
paid orders; its money is the total paid; and, because an average cannot be checked by adding averages up, every
status has an average and it is the money divided by the orders. Counts must be exactly equal. Money must agree to
the cent: `abs(a - b) < 0.005`, never `==`.

**When the five lines say `PASS`, the table has passed every check written here.** A check tests only what it was
written to test: before you call it done, say what one row of your table is, and read its rows.

In [ ]:
# The check itself is written for you, in the cell *The checks* near the top.
check_e()

## The checkpoint

At minute 33 the last ten minutes begin (README), finished or not. **Before you leave**, the checkpoint: the keyboard
**saves this notebook** and uploads `notebooks/report.ipynb` to the lab's Moodle checkpoint; everyone else pastes the
*Your team* line from the top of the notebook as their submission text. Nothing else is handed in.

**Once section E passes, choose either of the next two sections**, if there is time; neither comes first.

## If there is time: one more delivered order

*Block 3, cells 9 and 10 (the rows with no partner), copied at the end of `lecture.ipynb`.*

`orders` has one more delivered order than your table (compare A2 with `status_summary`). Find it in `orders`: which order is
it, and why does your table not count it? The what-if at minute 33 is about this order.

In [ ]:
# The delivered order in orders that your table does not count.

## Your own question

*After section E passes, instead of the section above or after it. Nothing checks it.*

Pick one, say what one row of your query is before you type, guess the answer, then write it in the cell below:

1. Which payment type brings in the most money, and in how many orders? Does this question need a join at all?
2. How many delivered orders were paid, at least in part, with a voucher?
3. Which order status has the most payment rows per paid order, on average? (One row per order first.)

At minute 33 it can be the result you show the team next to you first.

In [ ]:
# Your question, its number, and your guess before you ran it.